In [0]:
from pyspark.sql import functions as F

def standardize_city(column):
    city = F.lower(F.trim(column))

    return (
        F.when(city.isNull() | (city == ""), "Unknown")
        .when(city.isin("prishtina", "prishtinë"), "Prishtinë")
        .when(city.isin("gjakove", "gjakovë"), "Gjakovë")
        .when(city.isin("peja", "pejë"), "Pejë")
        .when(city.isin("mitrovice", "mitrovicë"), "Mitrovicë")
        .when(city.isin("vushtrri", "vushtrria"), "Vushtrri")
        .when(city == "prizren", "Prizren")
        .when(city == "gjilan", "Gjilan")
        .when(city == "ferizaj", "Ferizaj")
        .otherwise(F.initcap(F.trim(column)))
    )

ORDERS_PATH = "/Volumes/workspace/ecommerce_dataset/ecommerce/orders_raw.csv"
CUSTOMERS_PATH = "/Volumes/workspace/ecommerce_dataset/ecommerce/customers_raw.csv"

orders_raw = spark.read.csv(ORDERS_PATH, header=True, inferSchema=True)
customers_raw = spark.read.csv(CUSTOMERS_PATH, header=True, inferSchema=True)

raw_order_count = orders_raw.count()
raw_customer_count = customers_raw.count()

print(f"Raw orders loaded: {raw_order_count:,}")
print(f"Raw customers loaded: {raw_customer_count:,}")

In [0]:
orders_working = (
    orders_raw
    .withColumn("_quantity_numeric", F.expr("try_cast(quantity as int)"))
    .withColumn("_unit_price_numeric", F.expr("try_cast(unit_price as double)"))
    .withColumn("_order_date_parsed", F.expr("try_cast(order_date as date)"))
)

print("Working order DataFrame prepared.")

In [0]:
orders_invalid = orders_working.filter(
    F.col("order_id").isNull() |
    F.col("customer_id").isNull() |
    F.col("_order_date_parsed").isNull() |
    F.col("_quantity_numeric").isNull() |
    F.col("_unit_price_numeric").isNull() |
    (F.col("_quantity_numeric") <= 0) |
    (F.col("_unit_price_numeric") <= 0)
)

invalid_order_count = orders_invalid.count()

missing_order_id_count = orders_working.filter(F.col("order_id").isNull()).count()
missing_order_customer_id_count = orders_working.filter(F.col("customer_id").isNull()).count()
invalid_order_date_count = orders_working.filter(F.col("_order_date_parsed").isNull()).count()
invalid_order_quantity_count = orders_working.filter(F.col("_quantity_numeric").isNull() | (F.col("_quantity_numeric") <= 0)).count()
invalid_order_price_count = orders_working.filter(F.col("_unit_price_numeric").isNull() | (F.col("_unit_price_numeric") <= 0)).count()

print(f"Invalid order records identified: {invalid_order_count:,}")

print("\nInvalid order breakdown:")
print(f"Missing order_id: {missing_order_id_count:,}")
print(f"Missing customer_id: {missing_order_customer_id_count:,}")
print(f"Invalid/missing order_date: {invalid_order_date_count:,}")
print(f"Invalid/missing quantity: {invalid_order_quantity_count:,}")
print(f"Invalid/missing unit_price: {invalid_order_price_count:,}")

if invalid_order_count > 0:
    display(orders_invalid.limit(20))

In [0]:
orders_valid = orders_working.filter(
    F.col("order_id").isNotNull() &
    F.col("customer_id").isNotNull() &
    F.col("_order_date_parsed").isNotNull() &
    F.col("_quantity_numeric").isNotNull() &
    F.col("_unit_price_numeric").isNotNull() &
    (F.col("_quantity_numeric") > 0) &
    (F.col("_unit_price_numeric") > 0)
)

before_exact_dedup = orders_valid.count()
orders_valid = orders_valid.dropDuplicates()
after_exact_dedup = orders_valid.count()

exact_order_duplicates_removed = before_exact_dedup - after_exact_dedup

print(f"Exact duplicate order records removed: {exact_order_duplicates_removed:,}")

In [0]:
duplicate_order_ids = orders_valid.groupBy("order_id").count().filter(F.col("count") > 1).select("order_id")
conflicting_order_id_count = duplicate_order_ids.count()

print(f"Order IDs still duplicated after exact duplicate removal: {conflicting_order_id_count:,}")

if conflicting_order_id_count > 0:
    print("These records are excluded because there is no reliable rule for choosing which occurrence is authoritative.")
    display(orders_valid.join(duplicate_order_ids, on="order_id", how="inner").orderBy("order_id"))

orders_valid = orders_valid.join(duplicate_order_ids, on="order_id", how="left_anti")

In [0]:
orders_clean = (
    orders_valid
    .withColumn("order_date", F.col("_order_date_parsed"))
    .withColumn("quantity", F.col("_quantity_numeric"))
    .withColumn("unit_price", F.col("_unit_price_numeric"))
    .withColumn("city", standardize_city(F.col("city")))
    .withColumn(
        "product_category",
        F.when(F.col("product_category").isNull() | (F.trim(F.col("product_category")) == ""), "Unknown")
        .otherwise(F.initcap(F.trim(F.col("product_category"))))
    )
    .withColumn("_status_normalized", F.lower(F.trim(F.col("status"))))
    .withColumn(
        "status",
        F.when(F.col("_status_normalized").isin("complete", "completed"), "completed")
        .when(F.col("_status_normalized").isin("cancel", "canceled", "cancelled"), "cancelled")
        .when(F.col("_status_normalized").isNull() | (F.col("_status_normalized") == ""), "unknown")
        .otherwise(F.col("_status_normalized"))
    )
    .withColumn("_payment_normalized", F.lower(F.trim(F.col("payment_method"))))
    .withColumn(
        "payment_method",
        F.when(F.col("_payment_normalized").isNull() | (F.col("_payment_normalized") == ""), "Unknown")
        .when(F.col("_payment_normalized").contains("paypal"), "PayPal")
        .when(F.col("_payment_normalized").contains("bank"), "Bank Transfer")
        .when(F.col("_payment_normalized").contains("credit") | F.col("_payment_normalized").contains("card"), "Credit Card")
        .when(F.col("_payment_normalized").contains("cash"), "Cash")
        .otherwise(F.initcap(F.trim(F.col("payment_method"))))
    )
    .withColumn("total_amount", F.round(F.col("quantity") * F.col("unit_price"), 2))
    .drop("_quantity_numeric", "_unit_price_numeric", "_order_date_parsed", "_status_normalized", "_payment_normalized")
)

print(f"Clean orders after validation: {orders_clean.count():,}")
display(orders_clean.limit(10))

In [0]:
orders_correction_comparison = orders_raw.alias("raw").join(orders_clean.alias("clean"), on="order_id", how="inner")

status_changed = F.coalesce(F.col("raw.status"), F.lit("")) != F.coalesce(F.col("clean.status"), F.lit(""))
payment_changed = F.coalesce(F.col("raw.payment_method"), F.lit("")) != F.coalesce(F.col("clean.payment_method"), F.lit(""))
city_changed = F.coalesce(F.col("raw.city"), F.lit("")) != F.coalesce(F.col("clean.city"), F.lit(""))
category_changed = F.coalesce(F.col("raw.product_category"), F.lit("")) != F.coalesce(F.col("clean.product_category"), F.lit(""))

status_corrected_count = orders_correction_comparison.filter(status_changed).select("order_id").distinct().count()
payment_corrected_count = orders_correction_comparison.filter(payment_changed).select("order_id").distinct().count()
city_corrected_count = orders_correction_comparison.filter(city_changed).select("order_id").distinct().count()
category_corrected_count = orders_correction_comparison.filter(category_changed).select("order_id").distinct().count()

corrected_order_count = (
    orders_correction_comparison
    .filter(status_changed | payment_changed | city_changed | category_changed)
    .select("order_id")
    .distinct()
    .count()
)

print("Order standardization corrections:")
print(f"Order records requiring at least one correction: {corrected_order_count:,}")
print(f"Status values changed: {status_corrected_count:,}")
print(f"Payment-method values changed: {payment_corrected_count:,}")
print(f"City values changed: {city_corrected_count:,}")
print(f"Product-category values changed: {category_corrected_count:,}")

In [0]:
missing_customer_id_count = customers_raw.filter(F.col("customer_id").isNull()).count()
missing_customer_name_count = customers_raw.filter(F.col("customer_name").isNull()).count()
missing_customer_email_count = customers_raw.filter(F.col("email").isNull()).count()
missing_customer_city_count = customers_raw.filter(F.col("city").isNull()).count()

malformed_customer_emails = customers_raw.filter(
    F.col("email").isNotNull() &
    ~F.col("email").rlike(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")
)

malformed_customer_email_count = malformed_customer_emails.count()

print("CUSTOMER DATA-QUALITY ISSUES")
print(f"Missing customer_id: {missing_customer_id_count:,}")
print(f"Missing customer_name: {missing_customer_name_count:,}")
print(f"Missing email: {missing_customer_email_count:,}")
print(f"Malformed non-null email: {malformed_customer_email_count:,}")
print(f"Missing city: {missing_customer_city_count:,}")

if malformed_customer_email_count > 0:
    print("\nSample malformed emails:")
    display(malformed_customer_emails.select("customer_id", "customer_name", "email").limit(20))

In [0]:
customers_valid = customers_raw.filter(F.col("customer_id").isNotNull())

before_customer_exact_dedup = customers_valid.count()
customers_valid = customers_valid.dropDuplicates()
after_customer_exact_dedup = customers_valid.count()

exact_customer_duplicates_removed = before_customer_exact_dedup - after_customer_exact_dedup

print(f"Exact duplicate customer records removed: {exact_customer_duplicates_removed:,}")

duplicate_customer_ids = customers_valid.groupBy("customer_id").count().filter(F.col("count") > 1).select("customer_id")
conflicting_customer_id_count = duplicate_customer_ids.count()

print(f"Customer IDs still duplicated after exact duplicate removal: {conflicting_customer_id_count:,}")

if conflicting_customer_id_count > 0:
    print("Conflicting customer IDs are excluded because they cannot be matched reliably.")
    display(customers_valid.join(duplicate_customer_ids, on="customer_id", how="inner").orderBy("customer_id"))

customers_valid = customers_valid.join(duplicate_customer_ids, on="customer_id", how="left_anti")

In [0]:
customers_clean = (
    customers_valid
    .withColumn(
        "customer_name",
        F.when(F.col("customer_name").isNull() | (F.trim(F.col("customer_name")) == ""), "Unknown")
        .otherwise(F.trim(F.col("customer_name")))
    )
    .withColumn(
        "email",
        F.when(F.col("email").isNull(), F.lit(None))
        .when(F.col("email").rlike(r"^[^@\s]+@[^@\s]+\.[^@\s]+$"), F.lower(F.trim(F.col("email"))))
        .otherwise(F.lit(None))
    )
    .withColumn("city", standardize_city(F.col("city")))
    .withColumn("_customer_type_normalized", F.lower(F.trim(F.col("customer_type"))))
    .withColumn(
        "customer_type",
        F.when(F.col("_customer_type_normalized") == "vip", "VIP")
        .when(F.col("_customer_type_normalized") == "business", "Business")
        .when(F.col("_customer_type_normalized") == "retail", "Retail")
        .when(F.col("_customer_type_normalized").isNull() | (F.col("_customer_type_normalized") == ""), "Unknown")
        .otherwise(F.initcap(F.col("_customer_type_normalized")))
    )
    .drop("_customer_type_normalized")
)

print(f"Clean customers: {customers_clean.count():,}")
display(customers_clean.limit(10))

In [0]:
valid_customer_ids = customers_clean.select("customer_id").distinct()

unmatched_customer_ids = (
    orders_clean
    .select("customer_id")
    .distinct()
    .join(valid_customer_ids, on="customer_id", how="left_anti")
)

unmatched_customer_id_count = unmatched_customer_ids.count()

orders_without_valid_customer = orders_clean.join(unmatched_customer_ids, on="customer_id", how="inner")
unmatched_order_count = orders_without_valid_customer.count()

print(f"Customer IDs referenced by trusted orders but absent from trusted customers: {unmatched_customer_id_count:,}")
print(f"Order records affected by unmatched customer references: {unmatched_order_count:,}")

if unmatched_order_count > 0:
    display(orders_without_valid_customer.limit(20))

In [0]:
orders_silver = orders_clean
customers_silver = customers_clean

clean_order_count = orders_silver.count()
clean_customer_count = customers_silver.count()

print("TRUSTED DATASETS CREATED")
print(f"orders_silver: {clean_order_count:,} records")
print(f"customers_silver: {clean_customer_count:,} records")

print("\nOrders schema:")
orders_silver.printSchema()

print("\nCustomers schema:")
customers_silver.printSchema()

In [0]:
duplicate_order_ids_after_cleaning = orders_silver.groupBy("order_id").count().filter(F.col("count") > 1).count()
duplicate_customer_ids_after_cleaning = customers_silver.groupBy("customer_id").count().filter(F.col("count") > 1).count()

invalid_quantity_after_cleaning = orders_silver.filter(F.col("quantity").isNull() | (F.col("quantity") <= 0)).count()
invalid_price_after_cleaning = orders_silver.filter(F.col("unit_price").isNull() | (F.col("unit_price") <= 0)).count()

missing_order_id_after_cleaning = orders_silver.filter(F.col("order_id").isNull()).count()
missing_customer_id_after_cleaning = orders_silver.filter(F.col("customer_id").isNull()).count()
invalid_order_date_after_cleaning = orders_silver.filter(F.col("order_date").isNull()).count()

invalid_total_amount_after_cleaning = orders_silver.filter(
    F.col("total_amount") != F.round(F.col("quantity") * F.col("unit_price"), 2)
).count()

print("SILVER VALIDATION")
print(f"Duplicate order IDs: {duplicate_order_ids_after_cleaning}")
print(f"Duplicate customer IDs: {duplicate_customer_ids_after_cleaning}")
print(f"Invalid quantities: {invalid_quantity_after_cleaning}")
print(f"Invalid unit prices: {invalid_price_after_cleaning}")
print(f"Missing order IDs: {missing_order_id_after_cleaning}")
print(f"Missing customer IDs in orders: {missing_customer_id_after_cleaning}")
print(f"Invalid/missing order dates: {invalid_order_date_after_cleaning}")
print(f"Incorrect total_amount values: {invalid_total_amount_after_cleaning}")

assert duplicate_order_ids_after_cleaning == 0
assert duplicate_customer_ids_after_cleaning == 0
assert invalid_quantity_after_cleaning == 0
assert invalid_price_after_cleaning == 0
assert missing_order_id_after_cleaning == 0
assert missing_customer_id_after_cleaning == 0
assert invalid_order_date_after_cleaning == 0
assert invalid_total_amount_after_cleaning == 0

print("\nAll trusted-data validation checks passed.")

unexpected_trusted_statuses = (
    orders_silver
    .filter(~F.col("status").isin("completed", "pending", "cancelled", "refunded"))
    .groupBy("status")
    .count()
    .orderBy(F.desc("count"))
)

print("\nUnexpected trusted status values:")
display(unexpected_trusted_statuses)

In [0]:
orders_silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.ecommerce_dataset.orders_silver")
customers_silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.ecommerce_dataset.customers_silver")

print("Silver tables saved successfully.")

In [0]:
orders_removed = raw_order_count - clean_order_count
customers_removed = raw_customer_count - clean_customer_count

order_cleaning_success_rate = clean_order_count / raw_order_count * 100
customer_cleaning_success_rate = clean_customer_count / raw_customer_count * 100

print("\nORDERS")
print(f"Raw order records: {raw_order_count:,}")
print(f"Invalid order records identified: {invalid_order_count:,}")
print(f"Exact duplicate records removed: {exact_order_duplicates_removed:,}")
print(f"Conflicting duplicate IDs: {conflicting_order_id_count:,}")
print(f"Trusted order records: {clean_order_count:,}")
print(f"Total order records removed: {orders_removed:,}")
print(f"Cleaning success rate: {order_cleaning_success_rate:.2f}%")
print(f"Order records requiring at least one correction: {corrected_order_count:,}")

print("\nCORRECTIONS")
print(f"Status values standardized: {status_corrected_count:,}")
print(f"Payment methods standardized: {payment_corrected_count:,}")
print(f"City values standardized: {city_corrected_count:,}")
print(f"Product categories standardized: {category_corrected_count:,}")

print("\nCUSTOMERS")
print(f"Raw customer records: {raw_customer_count:,}")
print(f"Missing customer names identified: {missing_customer_name_count:,}")
print(f"Missing emails identified: {missing_customer_email_count:,}")
print(f"Malformed emails identified: {malformed_customer_email_count:,}")
print(f"Missing cities identified: {missing_customer_city_count:,}")
print(f"Exact duplicate records removed: {exact_customer_duplicates_removed:,}")
print(f"Conflicting duplicate IDs: {conflicting_customer_id_count:,}")
print(f"Trusted customer records: {clean_customer_count:,}")
print(f"Total customer records removed: {customers_removed:,}")
print(f"Cleaning success rate: {customer_cleaning_success_rate:.2f}%")

print("\nRELATIONSHIPS")
print(f"Trusted orders without a matching trusted customer: {unmatched_order_count:,}")